# Run_Training notebook: Flex at 32x32 with periodic testing

This notebook mirrors `Run_Training.py` for the AiFnet single-parameter dataset.

Changes requested here:
- use `Backbone.Flex`
- use `32x32`
- run a held-out test pass every `200` epochs

The training recipe stays aligned with `Run_Training.py` and `Trainer.py`:
- `AdamW`
- cosine LR schedule with warmup
- EMA model tracking
- `CosSchDiffuser`
- epsilon prediction loss (`type="e"`)


In [ ]:
from __future__ import annotations

import math
import os
import random
import sys
from collections import OrderedDict
from contextlib import contextmanager
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd() / 'FoilDIff', Path.cwd().parent / 'FoilDIff']
    for candidate in candidates:
        if (candidate / 'Run_Training.py').exists() and (candidate / 'Backbone.py').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the FoilDIff repo root from the current working directory.')


REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import Backbone
import Diffuser as diff
from AiFnet.airfoil_diffusion.airfoil_datasets import AirfoilDataset, FileDataFiles

print(f'repo root: {REPO_ROOT}')


In [ ]:
SEED = 42
IMAGE_SIZE = 32
BATCH_SIZE = 10
NOISE_STEPS = 200
EPOCHS = 2000
LR = 1e-4
EMA_DECAY = 0.9999
TRAIN_TYPE = 'e'
TEST_EVERY = 200
CHECKPOINT_EVERY = 500
TEST_CASE_INDEX = 0
TEST_BATCHES = 4
TEST_SEED = 123

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TRAIN_LIST = './AiFnet/datasets/1_parameter/train_cases.txt'
TEST_LIST = './AiFnet/datasets/1_parameter/test_cases.txt'
DATA_BASE = './AiFnet/datasets/1_parameter/data/'
TRAINER_SAVE_PATH = './checkpoints/model_checkpoint'
MODEL_DIR = REPO_ROOT / 'models' / '32'
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)


@contextmanager
def temporary_seed(seed: int):
    cpu_state = torch.get_rng_state()
    np_state = np.random.get_state()
    py_state = random.getstate()
    cuda_states = None
    if torch.cuda.is_available():
        cuda_states = torch.cuda.get_rng_state_all()
    set_seed(seed)
    try:
        yield
    finally:
        torch.set_rng_state(cpu_state)
        np.random.set_state(np_state)
        random.setstate(py_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)


set_seed(SEED)
print(f'device: {DEVICE}')


In [ ]:
train_dataset = AirfoilDataset(
    FileDataFiles(TRAIN_LIST, base_path=DATA_BASE),
    data_size=IMAGE_SIZE,
)

test_dataset = AirfoilDataset(
    FileDataFiles(TEST_LIST, base_path=DATA_BASE),
    data_size=IMAGE_SIZE,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

fixed_test_condition_cpu, fixed_test_target_cpu, fixed_test_meta = test_dataset[TEST_CASE_INDEX]
fixed_test_condition = fixed_test_condition_cpu.unsqueeze(0).to(DEVICE)
fixed_test_target = fixed_test_target_cpu.unsqueeze(0).to(DEVICE)

print(f'train dataset size: {len(train_dataset)}')
print(f'test dataset size: {len(test_dataset)}')
print(f'train batches per epoch: {len(train_loader)}')
print(f'fixed test case: {fixed_test_meta["file_name"]}')
print('fixed condition shape:', tuple(fixed_test_condition.shape))
print('fixed target shape:', tuple(fixed_test_target.shape))


In [ ]:
def get_cosine_lambda(initial_lr: float, final_lr: float, epochs: int, warmup_epoch: int):
    def cosine_lambda(idx_epoch: int) -> float:
        if idx_epoch < warmup_epoch:
            return idx_epoch / warmup_epoch
        return 1 - (1 - (math.cos((idx_epoch - warmup_epoch) / (epochs - warmup_epoch) * math.pi) + 1) / 2) * (1 - final_lr / initial_lr)

    return cosine_lambda


def checkpoint_save(model: torch.nn.Module, loss: float, l_epoch: int, save_path: str, parameterization: str = 'e') -> None:
    base_dir = os.path.dirname(save_path)
    os.makedirs(base_dir, exist_ok=True)

    loss_file = os.path.join(base_dir, 'loss.csv')
    with open(loss_file, 'a') as f:
        f.write(f'{l_epoch},{loss:.6f}\n')

    model_name = model.__class__.__name__
    model_dir = os.path.join(base_dir, f'{model_name}_epoch_{l_epoch}_TParam_{parameterization}_loss_{loss:.4f}')
    os.makedirs(model_dir, exist_ok=True)

    checkpoint_file = os.path.join(model_dir, 'model.pth')
    torch.save(model.state_dict(), checkpoint_file)
    print(f'model checkpoint saved to {checkpoint_file}')

    arch_file = os.path.join(model_dir, 'architecture.txt')
    with open(arch_file, 'w') as f:
        f.write(str(model))


def update_ema(ema_model: torch.nn.Module, model: torch.nn.Module, decay: float = 0.999) -> None:
    ema_params = OrderedDict(ema_model.named_parameters())
    model_params = OrderedDict(model.named_parameters())
    for name, param in model_params.items():
        if name in ema_params:
            ema_params[name].data.mul_(decay).add_(param.data, alpha=1 - decay)


In [ ]:
model = Backbone.Flex(size=IMAGE_SIZE, noise_steps=NOISE_STEPS).to(DEVICE)
ema_model = deepcopy(model).to(DEVICE)
ema_model.eval()

diffuser = diff.CosSchDiffuser(steps=NOISE_STEPS, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    get_cosine_lambda(initial_lr=LR, final_lr=1e-5, epochs=EPOCHS, warmup_epoch=100),
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'model: {model.__class__.__name__}')
print(f'trainable parameters: {trainable_params:,}')
print(f'optimizer: {optimizer.__class__.__name__}(lr={LR})')
print(f'diffuser: {diffuser.name}, steps={diffuser.steps}')


In [ ]:
def train_step(model: torch.nn.Module, batch, diffuser: diff.Diffuser, device: torch.device, train_type: str = 'e') -> torch.Tensor:
    condition, targets, _meta = batch
    condition = condition.to(device)
    targets = targets.to(device)

    batch_size = condition.size(0)
    t = torch.randint(0, diffuser.steps, (batch_size,), dtype=torch.long, device=device)
    noise = torch.randn_like(targets)

    if train_type == 'e':
        noisy_xt = diffuser.forward_diffusion(targets, t, noise)
        prediction = model(noisy_xt, t, condition)
        loss = F.mse_loss(prediction, noise)
    elif train_type == 'x':
        noisy_xt = diffuser.forward_diffusion(targets, t, noise)
        prediction = model(noisy_xt, t, condition)
        loss = F.mse_loss(prediction, targets)
    elif train_type == 'v':
        velocity = diffuser.calculate_velocity(targets, t, noise)
        prediction = model(velocity, t, condition)
        loss = F.mse_loss(prediction, velocity)
    else:
        raise ValueError(f'Unknown training type: {train_type}')

    del batch, condition, targets
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return loss


def evaluate_noise_loss(model: torch.nn.Module, loader: DataLoader, diffuser: diff.Diffuser, device: torch.device, num_batches: int = 4) -> float:
    was_training = model.training
    model.eval()
    losses = []
    with torch.no_grad():
        for idx, batch in enumerate(loader):
            if idx >= num_batches:
                break
            losses.append(float(train_step(model, batch, diffuser, device, train_type=TRAIN_TYPE).item()))
    if was_training:
        model.train()
    return float(sum(losses) / len(losses)) if losses else float('nan')


def ddpm_sample_from_noise_safe(model: torch.nn.Module, diffuser: diff.Diffuser, condition: torch.Tensor, show_progress: bool = False) -> torch.Tensor:
    # Run_Training.py never samples during training. For notebook testing, keep DDPM
    # sampling but start at steps-1 because Backbone.Flex(noise_steps=200) cannot
    # embed t=200 while Diffuser.sample_from_noise starts there.
    with torch.no_grad():
        x_t = torch.randn_like(condition)
        t_now = torch.full((x_t.shape[0],), diffuser.steps - 1, device=condition.device, dtype=torch.long)
        iterator = range(diffuser.steps - 1)
        if show_progress:
            iterator = tqdm(iterator, desc='DDPM test sampling', leave=False)
        for _ in iterator:
            t_pre = torch.clamp(t_now - 1, min=0)
            predicted_noise = model(x_t, t_now, condition)
            x_t = diffuser.DDPM_sample_step(x_t, t_now, t_pre, predicted_noise)
            t_now = t_pre
        return x_t


def channel_limits(array: np.ndarray) -> tuple[float, float]:
    vmin = float(array.min())
    vmax = float(array.max())
    if vmin == vmax:
        eps = 1e-6 if vmin == 0 else abs(vmin) * 1e-6
        vmin -= eps
        vmax += eps
    return vmin, vmax


def plot_prediction(target_tensor: torch.Tensor, prediction_tensor: torch.Tensor, epoch: int) -> None:
    target_np = target_tensor[0].detach().cpu().numpy()
    prediction_np = prediction_tensor[0].detach().cpu().numpy()
    error_np = np.abs(prediction_np - target_np)
    channel_names = ['Pressure', 'Velocity x', 'Velocity y']

    fig, axes = plt.subplots(3, 3, figsize=(12, 11))
    fig.suptitle(f'Fixed test case inference at epoch {epoch}', fontsize=16)

    for row, name in enumerate(channel_names):
        target_vmin, target_vmax = channel_limits(target_np[row])
        pred_vmin, pred_vmax = channel_limits(prediction_np[row])
        err_vmin, err_vmax = channel_limits(error_np[row])

        axes[row, 0].imshow(target_np[row], cmap='viridis', vmin=target_vmin, vmax=target_vmax)
        axes[row, 0].set_title(f'{name} target')
        axes[row, 1].imshow(prediction_np[row], cmap='viridis', vmin=pred_vmin, vmax=pred_vmax)
        axes[row, 1].set_title(f'{name} prediction')
        axes[row, 2].imshow(error_np[row], cmap='magma', vmin=err_vmin, vmax=err_vmax)
        axes[row, 2].set_title(f'{name} abs error')
        for col in range(3):
            axes[row, col].axis('off')

    plt.tight_layout()
    plt.show()


def run_test(model: torch.nn.Module, diffuser: diff.Diffuser, test_loader: DataLoader, condition: torch.Tensor, target: torch.Tensor, epoch: int) -> dict:
    was_training = model.training
    model.eval()
    test_noise_loss = evaluate_noise_loss(model, test_loader, diffuser, DEVICE, num_batches=TEST_BATCHES)
    with temporary_seed(TEST_SEED):
        prediction = ddpm_sample_from_noise_safe(model, diffuser, condition, show_progress=False)
    sample_mse = float(F.mse_loss(prediction, target).item())
    sample_mae = float(F.l1_loss(prediction, target).item())
    if was_training:
        model.train()
    plot_prediction(target, prediction, epoch)
    return {
        'epoch': epoch,
        'test_noise_loss': test_noise_loss,
        'sample_mse': sample_mse,
        'sample_mae': sample_mae,
    }


In [ ]:
progress_bar = tqdm(total=EPOCHS * len(train_loader), desc='Training', dynamic_ncols=True)
loss_history = []
test_history = []

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()
        loss = train_step(model, batch, diffuser, DEVICE, train_type=TRAIN_TYPE)
        loss.backward()
        optimizer.step()
        update_ema(ema_model, model, decay=EMA_DECAY)

        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])
        progress_bar.update(1)

        del batch, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    epoch_loss /= len(train_loader)
    loss_history.append(epoch_loss)
    scheduler.step()

    epoch_idx = epoch + 1

    if epoch_idx % CHECKPOINT_EVERY == 0:
        checkpoint_save(model, loss_history[-1], epoch_idx, TRAINER_SAVE_PATH, parameterization=TRAIN_TYPE)

    if epoch_idx % TEST_EVERY == 0:
        result = run_test(model, diffuser, test_loader, fixed_test_condition, fixed_test_target, epoch_idx)
        test_history.append(result)
        print(
            f'epoch {epoch_idx:4d} | train_loss={epoch_loss:.6f} | '
            f'test_noise_loss={result["test_noise_loss"]:.6f} | '
            f'sample_mse={result["sample_mse"]:.6f} | '
            f'sample_mae={result["sample_mae"]:.6f}'
        )

progress_bar.close()
print('training complete.')
checkpoint_save(model, loss_history[-1], EPOCHS, TRAINER_SAVE_PATH, parameterization=TRAIN_TYPE)

final_model_path = MODEL_DIR / f'{model.__class__.__name__}.pth'
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    },
    final_model_path,
)
print(f'model saved to {final_model_path}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(1, len(loss_history) + 1), loss_history)
axes[0].set_title('Train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].grid(True)

if test_history:
    epochs = [entry['epoch'] for entry in test_history]
    test_noise_loss = [entry['test_noise_loss'] for entry in test_history]
    sample_mse = [entry['sample_mse'] for entry in test_history]
    axes[1].plot(epochs, test_noise_loss, marker='o', label='test noise loss')
    axes[1].plot(epochs, sample_mse, marker='s', label='sample mse')
    axes[1].set_title('Periodic test metrics')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Metric value')
    axes[1].grid(True)
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'No test history yet', ha='center', va='center')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

test_history
